In [26]:
import torch
import torchvision
import pandas as pd
import pathlib
from PIL import Image

from torch.utils.data import Dataset, DataLoader

In [27]:
root = pathlib.Path(pathlib.Path.home() / "Downloads" / "csiro-biomass")

In [28]:
test_data = pd.read_csv(root / "test.csv")

In [29]:
test_data.head()

,sample_id,image_path,target_name
0,ID1001187975__Dry_Clover_g,test/ID1001187975.jpg,Dry_Clover_g
1,ID1001187975__Dry_Dead_g,test/ID1001187975.jpg,Dry_Dead_g
2,ID1001187975__Dry_Green_g,test/ID1001187975.jpg,Dry_Green_g
3,ID1001187975__Dry_Total_g,test/ID1001187975.jpg,Dry_Total_g
4,ID1001187975__GDM_g,test/ID1001187975.jpg,GDM_g


In [62]:
class BiomassTestDataset(Dataset):
    def __init__(self, csv_path, root, transform=None):
        super().__init__()
        self.annotation = pd.read_csv(csv_path)
        self.root = pathlib.Path(root)
        self.transform = transform

        # self.tabular_cols = ["Pre_GSHH_NDVI", "Height_Ave_cm"]
        self.annotation[self.tabular_cols] = self.annotation[self.tabular_cols].fillna(0)

    def __len__(self):
        return len(self.annotation)

    def __getitem__(self, idx):
        row = self.annotation.iloc[idx]

        img_path = self.root / row["image_path"]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        tabular = torch.tensor(row[self.tabular_cols].values, dtype=torch.float32)

        sample_id = row["sample_id"]
        target_name = row["target_name"]

        return image, tabular, sample_id, target_name


In [63]:
from torchvision import transforms as T

test_transform = T.Compose([
    T.Resize((224, 224)),                 
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]) 
])

In [64]:
test_dataset = BiomassTestDataset(csv_path = root / "test.csv", root = root, transform = test_transform)

KeyError: "None of [Index(['Pre_GSHH_NDVI', 'Height_Ave_cm'], dtype='object')] are in the [columns]"

In [65]:
test_dataset = BiomassTestDataset(
    csv_path= root / "test.csv",
    root=root,
    transform=test_transform
)

KeyError: "None of [Index(['Pre_GSHH_NDVI', 'Height_Ave_cm'], dtype='object')] are in the [columns]"

In [40]:
test_loader = DataLoader(test_dataset,
                        shuffle = False,
                        batch_size = 1)
len(test_loader)

5

In [42]:
import torch
import torch.nn as nn
import torchvision

class CsiroHybridModel(nn.Module):
    def __init__(self, num_tabular_features: int = 9, hidden_units: int = 64, out_features: int = 5):
        super().__init__()

        self.cnn = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity() 

        self.tabular = nn.Sequential(
            nn.Linear(num_tabular_features, hidden_units),
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_units, 32),
            nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Linear(in_features + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, out_features)
        )

    def forward(self, img, tabular):
        img_feat = self.cnn(img)         # [B, 512]
        tab_feat = self.tabular(tabular) # [B, 32]
        x = torch.cat([img_feat, tab_feat], dim=1)
        return self.fc(x)


In [52]:
device = "mps" if torch.mps.is_available() else "cpu"

In [57]:
model = CsiroHybridModel()
model.load_state_dict(torch.load("best_model.pth"))
model.to(device)

CsiroHybridModel(
  (cnn): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, trac

In [61]:
model.eval()
rows = []

with torch.inference_mode():
    for image, tabular, sid, tname in test_loader:
        pred = model(image.to(device), tabular.to(device)).cpu().numpy()

        for i in range(len(sid)):
            target_idx = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"].index(tname[i])
            pred_value = float(pred[i][target_idx])
            rows.append({"sample_id": sid[i], "target": pred_value})

ValueError: not enough values to unpack (expected 4, got 3)